# Evolução da distribuição de ratings da Chess.com

Exploração dos dados processados pelo pipeline (`python main.py all`).

Pergunta central: **o significado competitivo de um rating mudou ao longo dos anos?**

Convenção: percentil clássico = % de jogadores com rating igual ou inferior.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

obs = pd.read_csv(ROOT / "data/processed/observations_clean.csv", parse_dates=["date"])
curves = pd.read_csv(ROOT / "data/processed/curves.csv")
targets = pd.read_csv(ROOT / "data/processed/targets.csv")
fixed = pd.read_csv(ROOT / "data/processed/fixed_ratings.csv")
obs["year"] = obs.date.dt.year
print(f"{len(obs):,} observações | {obs.year.min()}–{obs.year.max()}")
obs.groupby(["platform", "source"]).size()

## Cobertura: observações por ano × modalidade (Chess.com)

In [ ]:
obs[obs.platform == "chesscom"].pivot_table(
    index="year", columns="game_type", values="rating", aggfunc="size", fill_value=0
)

## Nuvem de observações + curvas ajustadas (Rapid)

In [ ]:
import matplotlib.pyplot as plt
from src.visualization import _new_fig, _ordered_colors

gt = "rapid"
d = obs[(obs.platform == "chesscom") & (obs.game_type == gt)]
c = curves[(curves.platform == "chesscom") & (curves.game_type == gt)]
years = sorted(c.year.unique())
colors = dict(zip(years, _ordered_colors(len(years))))

fig, ax = _new_fig((10, 6))
for year in years:
    dy = d[d.year == year]
    cy = c[c.year == year].sort_values("rating")
    ax.scatter(dy.rating, dy.percentile, s=10, alpha=0.35, color=colors[year])
    ax.plot(cy.rating, cy.percentile, color=colors[year], lw=2, label=str(year))
ax.set_xlabel("Rating"); ax.set_ylabel("Percentil")
ax.set_title(f"Chess.com {gt}: observações e curvas isotônicas por ano")
ax.legend(frameon=False, fontsize=8)
plt.show()

## A pergunta do projeto: rating 1000 (Rapid) ao longo do tempo

In [ ]:
f1000 = fixed[(fixed.platform == "chesscom") & (fixed.game_type == "rapid") & (fixed.rating == 1000)]
f1000[["year", "percentile_est", "pctl_lo", "pctl_hi", "n_obs", "low_confidence"]].sort_values("year")

## Quanto custa o Top 5% / Top 1% em cada ano?

In [ ]:
t = targets[(targets.platform == "chesscom") & (targets.game_type == "rapid")]
t.pivot_table(index="top_share", columns="year", values="rating_est").round(0)

## Figuras prontas

As figuras finais (PNG + HTML interativo) estão em `figures/` — geradas por
`python main.py viz`. O relatório completo com metodologia e limitações está
em `REPORT.md`.